# 5. Partitioning and file layout

The code of this module's lessons in [Learn Apache Spark by Doing](https://dataenglab.com/courses/learn-apache-spark-by-doing/), in lesson order, on Apache Spark 4.2.0 in local mode: one machine, no cluster.

Colab runs this notebook in your own Google account: sign in with Google to run the cells, and use *File → Save a copy in Drive* to keep your changes. Nothing runs on DataEngLab and nothing is installed on your computer.

Run the cells from top to bottom. Under a cell, the result the lesson shows is there to compare with yours.

Not available in Colab: the Spark UI (its port is not reachable from a notebook). Where a lesson sends you to it, run that part on your machine; here, `df.explain()` prints the plan.

In [ ]:
# Setup: Java 17+, PySpark 4.2.0 and the webshop data. About a minute the first time.
import os, re, subprocess, sys, urllib.request, zipfile

PYSPARK_VERSION = "4.2.0"
DATA_URL = 'https://github.com/dataeng-lab/learn-apache-spark-by-doing/raw/v2/datasets.zip'

if "google.colab" in sys.modules:
    version = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    major = re.search(r'version "(?:1\.)?(\d+)', version)
    if not major or int(major.group(1)) < 17:       # PySpark 4 does not start on Java 8 or 11
        subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless",
                       shell=True, check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"], check=True)

if not os.path.isdir("datasets"):
    if DATA_URL is None:
        raise RuntimeError("datasets/ is missing: upload the course's datasets folder in the Files panel")
    urllib.request.urlretrieve(DATA_URL, "datasets.zip")
    zipfile.ZipFile("datasets.zip").extractall(".")

print("datasets:", sorted(os.listdir("datasets")))

**Starting point.** The DataFrames and names the earlier modules built. Not part of the lesson text.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("webshop")
    .master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

from pyspark.sql.types import (
    DecimalType, IntegerType, StringType, StructField, StructType, TimestampType,
)

ORDERS_SCHEMA = StructType([
    StructField("order_id", IntegerType(), nullable=False),
    StructField("customer_id", IntegerType(), nullable=True),
    StructField("order_ts", TimestampType(), nullable=True),
    StructField("status", StringType(), nullable=True),
    StructField("amount", DecimalType(10, 2), nullable=True),
    StructField("currency", StringType(), nullable=True),
    StructField("updated_at", TimestampType(), nullable=True),
])
orders_schema = ORDERS_SCHEMA   # some lessons use this name

ORDERS = "datasets/orders.csv"
target = "warehouse/orders_by_month"      # where the partitioned write goes
many_files = "warehouse/orders_200_files"
one_file = "warehouse/orders_1_file"

## Three things called partitioning

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/three-things-called-partitioning/)

The lesson saves this as `runtime_partitions.py`.

In [ ]:
from pyspark.sql import functions as F

print(spark.conf.get("spark.sql.files.maxPartitionBytes"))

orders = spark.read.csv(ORDERS, header=True, schema=orders_schema)
print(orders.rdd.getNumPartitions())

(
    orders.withColumn("pid", F.spark_partition_id())
    .groupBy("pid")
    .agg(F.count("*").alias("rows"))
    .orderBy("pid")
    .show(truncate=False)
)

What the lesson shows. It comes from a longer version of this example, so the numbers match but your cell may print less:

```text
spark.sql.files.maxPartitionBytes = 134217728b
orders.csv on disk                = 296885 bytes
partitions when read              = 1

how many rows landed in each partition:
+---+----+
|pid|rows|
+---+----+
|0  |4235|
+---+----+
```

Not run here: a one-line preview: the order_year and order_month columns are added in 'Partitioned writes and file layout'.

```python
orders.write.partitionBy("order_year", "order_month").parquet(target)
```

The lesson saves this as `window_partitioning.py`.

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

orders = spark.read.csv(ORDERS, header=True, schema=orders_schema).repartition(8)

w = Window.partitionBy("status").orderBy(F.col("updated_at").desc())
ranked = orders.withColumn("rn", F.row_number().over(w))
ranked.explain()

What the lesson shows. It comes from a longer version of this example, so the numbers match but your cell may print less:

```text
physical partitions of the DataFrame: 8
distinct values of the window's partitionBy key (status): 5
physical partitions after the window: 1
```

## repartition and coalesce

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/repartition-and-coalesce/)

The lesson saves this as `repartition_vs_coalesce.py`.

In [ ]:
from pyspark.sql import functions as F

orders = spark.read.csv(ORDERS, header=True, schema=orders_schema).repartition(16)

rep = orders.repartition(4)
coa = orders.coalesce(4)

(
    rep.withColumn("pid", F.spark_partition_id())
    .groupBy("pid")
    .agg(F.count("*").alias("rows"))
    .orderBy("pid")
    .show(truncate=False)
)

What the lesson shows. It comes from a longer version of this example, so the numbers match but your cell may print less:

```text
starting partitions: 16
after repartition(4): 4
after coalesce(4)   : 4

--- repartition(4): a full shuffle, rows are spread evenly ---
+---+----+
|pid|rows|
+---+----+
|0  |1058|
|1  |1059|
|2  |1059|
|3  |1059|
+---+----+
```

In [ ]:
skewed = orders.repartition(8, "order_id").filter(F.col("order_id") < 4600)

What the lesson shows. It comes from a longer version of this example, so the numbers match but your cell may print less:

```text
input partitions (8, after a selective filter): [69, 89, 80, 75, 78, 71, 88, 92]
coalesce(2)   -> [315, 327]
repartition(2)-> [321, 321]
```

## Partitioned writes and file layout

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/partitioned-writes-and-file-layout/)

The lesson saves this as `partitioned_write.py`.

In [ ]:
from pyspark.sql import functions as F

orders = (
    spark.read.csv(ORDERS, header=True, schema=orders_schema)
    .withColumn("order_year", F.year("order_ts"))
    .withColumn("order_month", F.date_format("order_ts", "MM"))
)

orders.write.mode("overwrite").partitionBy("order_year", "order_month").parquet(target)

What the lesson shows. It comes from a longer version of this example, so the numbers match but your cell may print less:

```text
directory layout on disk:
  order_year=2026/order_month=03  (1 files)
  order_year=2026/order_month=04  (1 files)
  order_year=2026/order_month=05  (1 files)
  order_year=2026/order_month=06  (1 files)
```

In [ ]:
read_back = spark.read.parquet(target)
one_month = read_back.filter(
    (F.col("order_year") == 2026) & (F.col("order_month") == "03")
)
one_month.explain()

**Starting point for the next cell.** Not part of the lesson text.

In [ ]:
# Tonight's batch: one month of orders, standing in for the rows a nightly job rewrites.
todays_orders = orders.filter((F.col("order_year") == 2026) & (F.col("order_month") == "06"))

In [ ]:
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

(
    todays_orders.write.mode("overwrite")
    .partitionBy("order_year", "order_month")
    .parquet(target)
)

## The small files problem

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/the-small-files-problem/)

The lesson saves this as `small_files.py`.

In [ ]:
orders = spark.read.csv(ORDERS, header=True, schema=orders_schema)

orders.repartition(200).write.mode("overwrite").parquet(many_files)
orders.coalesce(1).write.mode("overwrite").parquet(one_file)

What the lesson shows. It comes from a longer version of this example, so the numbers match but your cell may print less:

```text
written from 200 partitions -> 200 parquet files
total size 545 KiB, average file 2.7 KiB
the same rows from 1 partition  -> 1 parquet file
total size 110 KiB

same data, 200 files vs 1: 5.0x the bytes,
because every Parquet file repeats the footer, schema and column metadata.
```